In [ ]:
# Cell 1: install Hunyuan3D and dependencies. Run this first in Colab with GPU.
import os, sys
from pathlib import Path

os.environ["PYOPENGL_PLATFORM"] = "egl"
os.environ["PYGLET_HEADLESS"] = "True"

REPO_DIR = Path("/content/Hunyuan3D-2")
if not REPO_DIR.exists():
    !git clone https://github.com/Tencent-Hunyuan/Hunyuan3D-2.git /content/Hunyuan3D-2

%cd /content/Hunyuan3D-2

!apt-get -qq update
!apt-get -qq install -y libegl1-mesa libgles2-mesa mesa-utils
!pip -q install -r requirements.txt
!pip -q install -e .
!pip -q install --upgrade PyOpenGL PyOpenGL_accelerate pyrender==0.1.45 trimesh==4.4.1 open_clip_torch==2.24.0

sys.path.insert(0, str(REPO_DIR))
print("Setup complete. Now run Cell 2.")


In [ ]:
# Cell 2: multisignal evaluator code. Run this once after setup.
"""
Hunyuan3D mesh evaluator — multi-signal reliable scoring.

The single biggest reliability problem with image->mesh scoring is that the
input photo is ONE unknown viewpoint, while the rendered mesh is N viewpoints.
Comparing the photo against all N renders pollutes the score with viewpoint
mismatch noise that has nothing to do with mesh quality.

Design:
  1. Render 24 views (3 elevations x 8 azimuths) for good coverage.
  2. Find the K=3 renders whose silhouettes best match the input photo.
     These are the "matched views" — same approximate viewpoint as the photo.
  3. Score CLIP/DINO/SigLIP semantic similarity ON THE MATCHED VIEWS only
     (the primary signal — viewpoint-isolated, so it actually measures mesh
     quality rather than viewpoint luck).
  4. Use the FULL view set for consistency / blob / diversity checks.
  5. Add a geometry sanity check (faces, components, watertight, aspect, hull).
  6. Run bootstrap stability (re-score on random view subsets) — measures
     score fragility independent of absolute value.
  7. Cross-signal agreement check (CLIP, DINO, SigLIP, IoU all agree?) ->
     reliability flag with explicit reasons.

Outputs:
  - JSON with per-image full breakdown (rank percentile included)
  - Preview grid with per-view metrics overlaid
  - Spotlight image: input photo next to its top-3 matched renders
  - Saved mesh OBJ per image
  - Sortable HTML report

Optional models (graceful fallback):
  - DINOv2 (shape-aware, complements CLIP's color bias)
  - SigLIP  (often stronger than CLIP for image-image — a third
    independent semantic signal so reliability triangulation is stronger)
"""

import gc
import html
import json
import logging
import os
from pathlib import Path

import numpy as np
import open_clip
import pyrender
import torch
import trimesh
from PIL import Image, ImageDraw

from hy3dgen.rembg import BackgroundRemover
from hy3dgen.shapegen import (
    DegenerateFaceRemover,
    FaceReducer,
    FloaterRemover,
    Hunyuan3DDiTFlowMatchingPipeline,
)


# ============================================================================
# Constants
# ============================================================================

MODEL_ID = "tencent/Hunyuan3D-2"
SEED = 2025
RENDER_SIZE = 384
STEPS = 30

# 3 elevations x 8 azimuths = 24 views. More coverage than 16, not painfully slow.
ELEVATIONS = [0.0, 0.4, 0.8]
N_AZIMUTHS = 8
N_VIEWS = len(ELEVATIONS) * N_AZIMUTHS

# Matched-viewpoint scoring: how many top-IoU views to anchor on.
MATCHED_K = 3

# Multi-scale silhouette IoU search (catches scale mismatch between input
# and render).
IOU_SCALES = [0.85, 1.0, 1.15]

# Bootstrap stability check.
BOOTSTRAP_N = 8
BOOTSTRAP_SUBSET = 12


# ============================================================================
# Calibration mappings — convert raw cosines / IoU to 0..100 scale.
# Different models have very different baseline ranges; each gets its own.
# ============================================================================

def score_from_range(x, low, high):
    return float(np.clip((x - low) / (high - low), 0.0, 1.0) * 100.0)


def map_clip_to_100(x):
    # Shape-only renders vs colored photos. Original (0.20, 0.55) was tuned
    # for textured comparison and pinned everything at the floor in practice.
    return score_from_range(x, 0.18, 0.42)


def map_dino_to_100(x):
    # DINOv2 features are more shape-aware; cosines typically 0.40-0.70 for
    # untextured renders vs photos.
    return score_from_range(x, 0.40, 0.72)


def map_siglip_to_100(x):
    # SigLIP image-image cosines tend to be a bit higher than CLIP's.
    return score_from_range(x, 0.28, 0.58)


def map_iou_to_100(x):
    return float(np.clip(x, 0.0, 1.0) * 100.0)


# ============================================================================
# Model loading
# ============================================================================

def setup_device():
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print("device:", device)
    if device == "cuda":
        print("gpu:", torch.cuda.get_device_name(0))
    return device


def load_models(device, use_dino=True, use_siglip=True):
    rembg = BackgroundRemover()

    shape_pipe = Hunyuan3DDiTFlowMatchingPipeline.from_pretrained(MODEL_ID)
    try:
        shape_pipe.to(device)
    except Exception:
        pass

    clip_model, _, clip_preprocess = open_clip.create_model_and_transforms(
        "ViT-H-14",
        pretrained="laion2b_s32b_b79k",
    )
    clip_model = clip_model.to(device).eval()
    print("clip: loaded ViT-H-14 (laion2b)")

    dino = None
    if use_dino:
        try:
            dino = torch.hub.load("facebookresearch/dinov2", "dinov2_vitl14").to(device).eval()
            print("dino: loaded dinov2_vitl14")
        except Exception as exc:
            print("dino: unavailable, continuing without it:", exc)

    siglip_model = None
    siglip_preprocess = None
    if use_siglip:
        try:
            siglip_model, _, siglip_preprocess = open_clip.create_model_and_transforms(
                "ViT-B-16-SigLIP-384",
                pretrained="webli",
            )
            siglip_model = siglip_model.to(device).eval()
            print("siglip: loaded ViT-B-16-SigLIP-384")
        except Exception as exc:
            print("siglip: unavailable, continuing without it:", exc)

    return rembg, shape_pipe, clip_model, clip_preprocess, dino, siglip_model, siglip_preprocess


# ============================================================================
# Embeddings
# ============================================================================

@torch.no_grad()
def clip_embed(pil_img, model, preprocess, device):
    x = preprocess(pil_img.convert("RGB")).unsqueeze(0).to(device)
    f = model.encode_image(x)
    return torch.nn.functional.normalize(f, dim=-1).squeeze(0)


@torch.no_grad()
def dino_embed(pil_img, dino, device, size=224):
    img = pil_img.convert("RGB").resize((size, size))
    arr = np.asarray(img, dtype=np.float32) / 255.0
    mean = np.array([0.485, 0.456, 0.406], dtype=np.float32)
    std = np.array([0.229, 0.224, 0.225], dtype=np.float32)
    arr = (arr - mean) / std
    x = torch.from_numpy(arr.transpose(2, 0, 1)).unsqueeze(0).to(device)
    f = dino(x)
    return torch.nn.functional.normalize(f, dim=-1).squeeze(0)


@torch.no_grad()
def siglip_embed(pil_img, model, preprocess, device):
    x = preprocess(pil_img.convert("RGB")).unsqueeze(0).to(device)
    f = model.encode_image(x)
    return torch.nn.functional.normalize(f, dim=-1).squeeze(0)


def cosine(a, b):
    return float((a * b).sum().clamp(-1, 1).item())


# ============================================================================
# Silhouette utilities
# ============================================================================

def to_mask(pil_img, threshold=0.5):
    arr = np.asarray(pil_img)
    if arr.ndim == 3 and arr.shape[-1] == 4:
        return arr[..., 3] > int(255 * threshold)
    if arr.ndim == 2:
        return arr < 245
    return np.any(arr[..., :3] < 245, axis=-1)


def crop_normalize_mask(mask, size=256):
    """Bounding-box crop + center on square + resize. Removes framing
    differences so two silhouettes from different cameras are comparable."""
    ys, xs = np.where(mask)
    if len(xs) == 0:
        return np.zeros((size, size), dtype=bool)
    y0, y1 = ys.min(), ys.max() + 1
    x0, x1 = xs.min(), xs.max() + 1
    cropped = mask[y0:y1, x0:x1]
    h, w = cropped.shape
    side = max(h, w)
    canvas = np.zeros((side, side), dtype=bool)
    yo = (side - h) // 2
    xo = (side - w) // 2
    canvas[yo:yo + h, xo:xo + w] = cropped
    pil = Image.fromarray(canvas.astype(np.uint8) * 255).resize((size, size), Image.NEAREST)
    return np.asarray(pil) > 127


def iou(a, b):
    inter = np.logical_and(a, b).sum()
    union = np.logical_or(a, b).sum()
    return float(inter) / float(union) if union > 0 else 0.0


def multi_scale_iou(input_mask, render_mask, scales=IOU_SCALES):
    """Try a few relative scales and return the best IoU. Catches the case
    where the mesh has the right shape but slightly wrong proportions —
    fixed-scale IoU would unfairly punish it."""
    h, w = input_mask.shape
    best = 0.0
    for s in scales:
        if abs(s - 1.0) < 1e-6:
            scaled = render_mask
        else:
            new_h, new_w = max(1, int(h * s)), max(1, int(w * s))
            pil = Image.fromarray(render_mask.astype(np.uint8) * 255).resize((new_w, new_h), Image.NEAREST)
            arr = np.asarray(pil) > 127
            # Center on canvas of original size
            canvas = np.zeros_like(input_mask)
            ys = (h - new_h) // 2
            xs = (w - new_w) // 2
            ys_dst = max(0, ys)
            xs_dst = max(0, xs)
            ys_src = max(0, -ys)
            xs_src = max(0, -xs)
            cy = min(new_h - ys_src, h - ys_dst)
            cx = min(new_w - xs_src, w - xs_dst)
            if cy > 0 and cx > 0:
                canvas[ys_dst:ys_dst + cy, xs_dst:xs_dst + cx] = arr[ys_src:ys_src + cy, xs_src:xs_src + cx]
            scaled = canvas
        v = iou(input_mask, scaled)
        if v > best:
            best = v
    return best


# ============================================================================
# Rendering
# ============================================================================

def _look_at(camera_pos, target=np.zeros(3, dtype=np.float32), up=np.array([0, 1, 0], dtype=np.float32)):
    camera_pos = np.array(camera_pos, dtype=np.float32)
    forward = target - camera_pos
    forward /= np.linalg.norm(forward) + 1e-8
    right = np.cross(up, forward)
    right /= np.linalg.norm(right) + 1e-8
    true_up = np.cross(forward, right)
    pose = np.eye(4, dtype=np.float32)
    pose[:3, :3] = np.stack([right, true_up, forward], axis=1)
    pose[:3, 3] = camera_pos
    return pose


def render_self_test(size=128):
    """Render a known-good icosphere and verify non-empty output. This is the
    single most useful guard against silent score-of-zero failures caused by
    a broken EGL / OpenGL context on Colab. Returns (ok, alpha_coverage)."""
    try:
        m = trimesh.creation.icosphere(subdivisions=2)
        scene = pyrender.Scene(bg_color=[255, 255, 255, 0], ambient_light=[0.3] * 3)
        scene.add(pyrender.Mesh.from_trimesh(m, smooth=False))
        cam = pyrender.PerspectiveCamera(yfov=np.pi / 3.0)
        light = pyrender.DirectionalLight(color=np.ones(3), intensity=2.5)
        pose = np.eye(4, dtype=np.float32); pose[2, 3] = 2.5
        scene.add(cam, pose=pose)
        scene.add(light, pose=pose)
        r = pyrender.OffscreenRenderer(viewport_width=size, viewport_height=size)
        color, _ = r.render(scene, flags=pyrender.RenderFlags.RGBA)
        r.delete()
        alpha = color[..., 3]
        coverage = float((alpha > 10).mean())
        return coverage > 0.02, coverage
    except Exception as exc:
        print(f"render_self_test: exception {exc}")
        return False, 0.0


def render_views(mesh, size=RENDER_SIZE):
    mesh = mesh.copy()
    if not mesh.is_empty:
        mesh.apply_translation(-mesh.centroid)
        scale = np.max(mesh.extents) + 1e-8
        mesh.apply_scale(1.0 / scale)

    scene = pyrender.Scene(bg_color=[255, 255, 255, 0], ambient_light=[0.25] * 3)
    scene.add(pyrender.Mesh.from_trimesh(mesh, smooth=False))
    cam = pyrender.PerspectiveCamera(yfov=np.pi / 3.0)
    light = pyrender.DirectionalLight(color=np.ones(3), intensity=2.5)
    renderer = pyrender.OffscreenRenderer(viewport_width=size, viewport_height=size)

    rgb_views, rgba_views, view_meta = [], [], []
    radius = 2.2
    for ele_idx, elev in enumerate(ELEVATIONS):
        for az_idx in range(N_AZIMUTHS):
            theta = 2 * np.pi * (az_idx / N_AZIMUTHS)
            cam_pos = [radius * np.cos(theta), elev, radius * np.sin(theta)]
            pose = _look_at(cam_pos)
            nc = scene.add(cam, pose=pose)
            nl = scene.add(light, pose=pose)
            color, _ = renderer.render(scene, flags=pyrender.RenderFlags.RGBA)
            rgba = Image.fromarray(color.astype(np.uint8)).convert("RGBA")
            rgba_views.append(rgba)
            bg = Image.new("RGB", rgba.size, (255, 255, 255))
            bg.paste(rgba, (0, 0), rgba.split()[-1])
            rgb_views.append(bg)
            view_meta.append({"elev": float(elev), "azimuth_deg": float(np.degrees(theta))})
            scene.remove_node(nc)
            scene.remove_node(nl)
    renderer.delete()
    return rgb_views, rgba_views, view_meta


# ============================================================================
# Geometry sanity
# ============================================================================

def geometry_report(mesh):
    if mesh is None or mesh.is_empty or len(mesh.faces) == 0:
        return {
            "face_count": 0, "vertex_count": 0, "component_count": 0,
            "is_watertight": False, "aspect_ratio": None, "hull_ratio": None,
            "geometry_score_100": 0.0,
        }

    face_count = int(len(mesh.faces))
    vertex_count = int(len(mesh.vertices))
    try:
        component_count = len(mesh.split(only_watertight=False))
    except Exception:
        component_count = 99

    face_score = float(np.clip(face_count / 5000.0, 0.0, 1.0))
    component_score = float(np.clip(1.0 / max(1, component_count), 0.0, 1.0))
    watertight_score = 1.0 if mesh.is_watertight else 0.45

    try:
        extent = np.asarray(mesh.extents, dtype=np.float32)
        aspect = float(extent.max() / (extent.min() + 1e-8))
        aspect_score = float(np.clip(3.0 / max(3.0, aspect), 0.0, 1.0))
    except Exception:
        aspect = None
        aspect_score = 0.5

    # Hull ratio: penalize BOTH near-1 (sphere/cube blob) and near-0
    # (paper-thin / broken).
    hull_ratio = None
    hull_score = 0.65
    try:
        if mesh.is_watertight:
            hv = mesh.convex_hull.volume
            if hv > 0:
                hull_ratio = float(abs(mesh.volume) / hv)
                blob_pen = max(0.0, hull_ratio - 0.95) * 5.0
                thin_pen = max(0.0, 0.05 - hull_ratio) * 10.0
                hull_score = float(np.clip(1.0 - blob_pen - thin_pen, 0.0, 1.0))
    except Exception:
        pass

    score = (
        0.25 * face_score
        + 0.25 * component_score
        + 0.20 * watertight_score
        + 0.15 * aspect_score
        + 0.15 * hull_score
    ) * 100.0

    return {
        "face_count": face_count,
        "vertex_count": vertex_count,
        "component_count": int(component_count),
        "is_watertight": bool(mesh.is_watertight),
        "aspect_ratio": aspect,
        "hull_ratio": hull_ratio,
        "geometry_score_100": float(np.clip(score, 0.0, 100.0)),
    }


# ============================================================================
# Aggregation
# ============================================================================

def trimmed_mean(values, trim=0.1):
    """Drop top/bottom `trim` fraction. Robust to outlier views from
    e.g. self-occlusion or pathological angles."""
    s = np.sort(np.asarray(values, dtype=np.float32))
    if len(s) == 0:
        return 0.0
    drop = int(len(s) * trim)
    if drop > 0 and len(s) > 2 * drop:
        s = s[drop:-drop]
    return float(s.mean())


def matched_view_score(per_view_scores_100, ious_raw, k=MATCHED_K):
    """Mean of per-view scores on the K renders whose silhouettes best
    match the input. This is the heart of the design: it isolates the
    viewpoint that the photo was taken from, so the resulting CLIP/DINO
    score actually measures mesh quality, not viewpoint luck."""
    ious = np.asarray(ious_raw)
    k = min(k, len(ious))
    top_idx = np.argsort(ious)[-k:]
    return float(np.mean([per_view_scores_100[i] for i in top_idx])), top_idx.tolist()


def per_signal_consistency(scores_100):
    # Calibrated for shape-only renders: per-view variance is naturally high,
    # so we only penalize when views are REALLY bad, not merely below average.
    s = np.asarray(scores_100, dtype=np.float32)
    spread = float(np.max(s) - np.min(s))
    low_share = float(np.mean(s < 25.0))
    penalty = 0.0
    penalty += np.clip(spread / 65.0, 0.0, 1.0) * 6.0
    penalty += low_share * 10.0
    return float(np.clip(penalty, 0.0, 20.0))


def multi_signal_consistency(per_signal_scores):
    pens = [per_signal_consistency(s) for s in per_signal_scores if s is not None and len(s) > 0]
    if not pens:
        return 0.0
    return float(np.clip(np.mean(pens) * 1.1, 0.0, 25.0))


def inter_view_diversity(view_embeds):
    if len(view_embeds) < 2:
        return 0.0
    sims = []
    for i in range(len(view_embeds)):
        for j in range(i + 1, len(view_embeds)):
            sims.append(cosine(view_embeds[i], view_embeds[j]))
    return float(1.0 - np.mean(sims))


def bootstrap_stability(per_view_signals, geometry_score, weights, subset=BOOTSTRAP_SUBSET, n=BOOTSTRAP_N, seed=0):
    """Re-score on random view subsets; high std = fragile metric."""
    rng = np.random.default_rng(seed)
    n_views = len(next(iter(per_view_signals.values())))
    subset = min(subset, n_views)

    scores = []
    for _ in range(n):
        idx = rng.choice(n_views, size=subset, replace=False)
        signal_means = {}
        for name, arr in per_view_signals.items():
            sub = np.asarray(arr)[idx]
            signal_means[name] = 0.4 * float(sub.max()) + 0.6 * float(sub.mean())
        visual = sum(weights[k] * v for k, v in signal_means.items())
        scores.append(0.72 * visual + 0.28 * geometry_score)

    arr = np.asarray(scores, dtype=np.float32)
    return float(arr.mean()), float(arr.std())


def reliability_flag(signal_scores, consistency, geometry_score, blob_penalty, bootstrap_std, matched_iou):
    """(flag, reasons[]). Thresholds loosened so realistic shape-only renders
    can actually reach 'high' reliability when the mesh is good."""
    available = [float(v) for v in signal_scores.values() if v is not None]
    spread = max(available) - min(available) if len(available) >= 2 else 0.0

    reasons = []
    if spread > 45:
        reasons.append("signals disagree")
    if consistency > 18:
        reasons.append("some views score much worse")
    if geometry_score < 35:
        reasons.append("weak mesh geometry")
    if blob_penalty > 12:
        reasons.append("views look too similar")
    if bootstrap_std > 8:
        reasons.append("score unstable across view subsets")
    if matched_iou < 0.25:
        reasons.append("no rendered viewpoint matches the input silhouette well")

    if not reasons:
        return "high", reasons
    if (spread <= 50 and consistency <= 20 and geometry_score >= 30
            and bootstrap_std <= 10 and matched_iou >= 0.20):
        return "medium", reasons
    return "low", reasons


def fragility_penalty(bootstrap_std, matched_iou):
    """Small score penalty for cases the reliability system already distrusts.

    Reliability remains the main warning. This only prevents a fragile score
    from ranking too highly when the matched viewpoint is weak or the metric
    swings a lot across view subsets.
    """
    bootstrap_penalty = float(np.clip((bootstrap_std - 6.0) / 8.0, 0.0, 1.0) * 8.0)
    viewpoint_penalty = float(np.clip((0.30 - matched_iou) / 0.20, 0.0, 1.0) * 10.0)
    return float(np.clip(bootstrap_penalty + viewpoint_penalty, 0.0, 18.0))


# ============================================================================
# Diagnostic outputs
# ============================================================================

def make_preview_grid(input_image, views, per_view_metrics, matched_idx, out_path):
    thumb_size = (192, 192)
    labeled = []

    first = input_image.copy().resize(thumb_size)
    ImageDraw.Draw(first).text((6, 6), "INPUT", fill=(255, 0, 0))
    labeled.append(first)

    matched_set = set(matched_idx)
    for idx, view in enumerate(views):
        im = view.copy().resize(thumb_size)
        # Highlight matched views with a red border
        if idx in matched_set:
            d = ImageDraw.Draw(im)
            for off in range(3):
                d.rectangle([off, off, thumb_size[0] - 1 - off, thumb_size[1] - 1 - off], outline=(255, 0, 0))
        d = ImageDraw.Draw(im)
        m = per_view_metrics[idx]
        d.text((6, 6), f"v{idx + 1}{' *' if idx in matched_set else ''}", fill=(255, 0, 0))
        d.text((6, 22), f"clip {m['clip']:.0f}", fill=(0, 0, 200))
        if m.get("dino") is not None:
            d.text((6, 38), f"dino {m['dino']:.0f}", fill=(0, 110, 0))
        if m.get("siglip") is not None:
            d.text((6, 54), f"sig  {m['siglip']:.0f}", fill=(180, 80, 0))
        d.text((6, 70), f"iou  {m['iou']:.0f}", fill=(160, 0, 160))
        labeled.append(im)

    cols = 6
    rows = int(np.ceil(len(labeled) / cols))
    grid = Image.new("RGB", (cols * thumb_size[0], rows * thumb_size[1]), "white")
    for idx, im in enumerate(labeled):
        x = (idx % cols) * thumb_size[0]
        y = (idx // cols) * thumb_size[1]
        grid.paste(im, (x, y))
    grid.save(out_path)


def make_spotlight(input_image, views, matched_idx, out_path):
    """Input photo next to its top-K matched renders. The single most useful
    diagnostic: a human can see at a glance whether the matched renders
    actually look like the input."""
    size = (320, 320)
    panels = [input_image.copy().resize(size)]
    ImageDraw.Draw(panels[0]).text((10, 10), "INPUT", fill=(255, 0, 0))
    for rank, idx in enumerate(matched_idx[::-1], 1):
        im = views[idx].copy().resize(size)
        ImageDraw.Draw(im).text((10, 10), f"matched #{rank} (v{idx + 1})", fill=(255, 0, 0))
        panels.append(im)
    out = Image.new("RGB", (size[0] * len(panels), size[1]), "white")
    for i, p in enumerate(panels):
        out.paste(p, (i * size[0], 0))
    out.save(out_path)


def write_html_report(results, out_path):
    rows = sorted(results, key=lambda r: r["final_score_100"], reverse=True)
    rows_html = []
    for rank, r in enumerate(rows, 1):
        reasons = "; ".join(r.get("review_reasons", [])) or "&mdash;"
        dino = "&mdash;" if r.get("dino_score_100") is None else f"{r['dino_score_100']:.1f}"
        siglip = "&mdash;" if r.get("siglip_score_100") is None else f"{r['siglip_score_100']:.1f}"
        rel = r["reliability"]
        color = {"high": "#2e7d32", "medium": "#f9a825", "low": "#c62828"}.get(rel, "#555")
        rows_html.append(
            f"<tr>"
            f"<td>{rank}</td>"
            f"<td>{html.escape(r['image'])}</td>"
            f"<td><b>{r['final_score_100']:.1f}</b></td>"
            f"<td style='color:{color};font-weight:bold'>{rel}</td>"
            f"<td>{r['clip_score_100']:.1f}</td>"
            f"<td>{dino}</td>"
            f"<td>{siglip}</td>"
            f"<td>{r['silhouette_score_100']:.1f}</td>"
            f"<td>{r['geometry_score_100']:.1f}</td>"
            f"<td>{r['bootstrap_std']:.2f}</td>"
            f"<td>{r.get('fragility_penalty', 0.0):.1f}</td>"
            f"<td>{html.escape(reasons)}</td>"
            f"<td><a href='{html.escape(Path(r['preview_path']).name)}'>preview</a> "
            f"<a href='{html.escape(Path(r['spotlight_path']).name)}'>spotlight</a></td>"
            f"</tr>"
        )
    html_doc = f"""<!doctype html>
<html><head><meta charset='utf-8'><title>Hunyuan3D evaluation</title>
<style>
body {{ font-family: system-ui, sans-serif; margin: 24px; }}
table {{ border-collapse: collapse; width: 100%; }}
th, td {{ border: 1px solid #ccc; padding: 6px 10px; text-align: left; }}
th {{ background: #f0f0f0; cursor: pointer; }}
tr:nth-child(even) {{ background: #fafafa; }}
small {{ color: #666; }}
</style></head><body>
<h1>Hunyuan3D mesh evaluation</h1>
<p><small>Sorted by final_score (descending). Click preview/spotlight links to inspect.
<br>Reliability = high means CLIP, DINO, SigLIP and silhouette IoU all agree on the score.
Low reliability = open the spotlight image and judge by eye.</small></p>
<table>
<thead><tr>
<th>#</th><th>image</th><th>final</th><th>reliability</th>
<th>clip</th><th>dino</th><th>siglip</th><th>iou</th>
<th>geom</th><th>boot.std</th><th>frag.pen</th><th>reasons</th><th>view</th>
</tr></thead>
<tbody>
{''.join(rows_html)}
</tbody></table>
</body></html>"""
    Path(out_path).write_text(html_doc, encoding="utf-8")


# ============================================================================
# Pipeline
# ============================================================================

@torch.no_grad()
def make_mesh_from_image(img_path, rembg, shape_pipe, device, steps=STEPS):
    img = Image.open(img_path).convert("RGB").resize((1024, 1024))
    img_fg = rembg(img)
    g = torch.Generator(device=device).manual_seed(SEED)
    out = shape_pipe(image=img_fg, num_inference_steps=steps, mc_algo="mc", generator=g)[0]
    out = FloaterRemover()(out)
    out = DegenerateFaceRemover()(out)
    out = FaceReducer()(out)
    if isinstance(out, trimesh.Trimesh):
        return out, img_fg
    if hasattr(out, "as_trimesh"):
        return out.as_trimesh(), img_fg
    return out, img_fg


def score_one(img_path, rembg_img, mesh,
              clip_model, clip_preprocess,
              dino,
              siglip_model, siglip_preprocess,
              device, out_dir):

    inp_rgba = rembg_img.convert("RGBA") if isinstance(rembg_img, Image.Image) else \
        Image.open(img_path).convert("RGBA")
    inp_white = Image.new("RGB", inp_rgba.size, (255, 255, 255))
    inp_white.paste(inp_rgba, (0, 0), inp_rgba.split()[-1])
    inp_mask = crop_normalize_mask(to_mask(inp_rgba))

    rgb_views, rgba_views, view_meta = render_views(mesh)
    n = len(rgb_views)

    inp_clip = clip_embed(inp_white, clip_model, clip_preprocess, device)
    inp_dino = dino_embed(inp_white, dino, device) if dino is not None else None
    inp_siglip = siglip_embed(inp_white, siglip_model, siglip_preprocess, device) if siglip_model is not None else None

    clip_raw, clip_scores = [], []
    dino_raw, dino_scores = [], []
    siglip_raw, siglip_scores = [], []
    iou_raw, iou_scores = [], []
    diversity_embeds = []

    for rgb, rgba in zip(rgb_views, rgba_views):
        v_clip = clip_embed(rgb, clip_model, clip_preprocess, device)
        c = cosine(inp_clip, v_clip)
        clip_raw.append(c)
        clip_scores.append(map_clip_to_100(c))

        if dino is not None:
            v_dino = dino_embed(rgb, dino, device)
            d = cosine(inp_dino, v_dino)
            dino_raw.append(d)
            dino_scores.append(map_dino_to_100(d))
            diversity_embeds.append(v_dino)
        else:
            diversity_embeds.append(v_clip)

        if siglip_model is not None:
            v_sig = siglip_embed(rgb, siglip_model, siglip_preprocess, device)
            s = cosine(inp_siglip, v_sig)
            siglip_raw.append(s)
            siglip_scores.append(map_siglip_to_100(s))

        render_mask = crop_normalize_mask(to_mask(rgba))
        iou_val = multi_scale_iou(inp_mask, render_mask)
        iou_raw.append(iou_val)
        iou_scores.append(map_iou_to_100(iou_val))

    clip_arr = np.asarray(clip_scores, dtype=np.float32)
    iou_arr = np.asarray(iou_scores, dtype=np.float32)
    iou_raw_arr = np.asarray(iou_raw, dtype=np.float32)
    dino_arr = np.asarray(dino_scores, dtype=np.float32) if dino_scores else None
    siglip_arr = np.asarray(siglip_scores, dtype=np.float32) if siglip_scores else None

    # ---- Diagnostic: catch blank renders before they silently produce 0 ----
    if float(iou_raw_arr.max()) < 0.01:
        raise RuntimeError(
            f"All 24 rendered views are empty for {Path(img_path).name}. "
            "This usually means the offscreen renderer (pyrender/EGL) is broken "
            "in this Colab session, or the generated mesh has no faces. "
            "Try: Runtime -> Restart runtime, then re-run Cell 1, 2, 3 in order. "
            f"(mesh face_count={len(mesh.faces)}, is_empty={mesh.is_empty})"
        )

    # ---- KEY: matched-viewpoint scoring ----
    # Find the K=3 renders whose silhouettes best match the input photo.
    # Score CLIP/DINO/SigLIP on those — viewpoint-isolated quality signal.
    matched_clip_score, matched_idx = matched_view_score(clip_arr, iou_raw_arr)
    matched_dino_score = None
    if dino_arr is not None:
        matched_dino_score, _ = matched_view_score(dino_arr, iou_raw_arr)
    matched_siglip_score = None
    if siglip_arr is not None:
        matched_siglip_score, _ = matched_view_score(siglip_arr, iou_raw_arr)

    # Silhouette: the IoU score on the matched views themselves (essentially
    # the best-matching view IoU) — calibration-free shape match.
    matched_iou_raw = float(np.mean([iou_raw_arr[i] for i in matched_idx]))
    silhouette_score_100 = map_iou_to_100(matched_iou_raw)

    # ---- Final visual score: matched-view fusion ----
    # Heavier weight on calibrated semantic signals (CLIP), iou as anchor.
    if dino_arr is None and siglip_arr is None:
        visual_score_100 = 0.65 * matched_clip_score + 0.35 * silhouette_score_100
        weights = {"clip": 0.65, "iou": 0.35}
        per_view_for_boot = {"clip": clip_arr, "iou": iou_arr}
    elif dino_arr is not None and siglip_arr is None:
        visual_score_100 = 0.45 * matched_clip_score + 0.25 * matched_dino_score + 0.30 * silhouette_score_100
        weights = {"clip": 0.45, "dino": 0.25, "iou": 0.30}
        per_view_for_boot = {"clip": clip_arr, "dino": dino_arr, "iou": iou_arr}
    elif dino_arr is None and siglip_arr is not None:
        visual_score_100 = 0.45 * matched_clip_score + 0.25 * matched_siglip_score + 0.30 * silhouette_score_100
        weights = {"clip": 0.45, "siglip": 0.25, "iou": 0.30}
        per_view_for_boot = {"clip": clip_arr, "siglip": siglip_arr, "iou": iou_arr}
    else:
        visual_score_100 = (
            0.35 * matched_clip_score
            + 0.20 * matched_dino_score
            + 0.20 * matched_siglip_score
            + 0.25 * silhouette_score_100
        )
        weights = {"clip": 0.35, "dino": 0.20, "siglip": 0.20, "iou": 0.25}
        per_view_for_boot = {"clip": clip_arr, "dino": dino_arr, "siglip": siglip_arr, "iou": iou_arr}

    geo = geometry_report(mesh)

    consistency = multi_signal_consistency([clip_arr, iou_arr, dino_arr, siglip_arr])
    diversity = inter_view_diversity(diversity_embeds)
    blob_penalty = float(np.clip((0.05 - diversity) * 500.0, 0.0, 20.0)) if diversity < 0.05 else 0.0

    boot_mean, boot_std = bootstrap_stability(
        per_view_for_boot, geo["geometry_score_100"], weights
    )

    fragility = fragility_penalty(boot_std, matched_iou_raw)
    final = (
        0.72 * visual_score_100
        + 0.28 * geo["geometry_score_100"]
        - consistency
        - blob_penalty
        - fragility
    )
    final = float(np.clip(final, 0.0, 100.0))

    signal_scores = {
        "clip": matched_clip_score,
        "dino": matched_dino_score,
        "siglip": matched_siglip_score,
        "iou": silhouette_score_100,
    }
    flag, reasons = reliability_flag(
        signal_scores, consistency, geo["geometry_score_100"],
        blob_penalty, boot_std, matched_iou_raw,
    )

    # ---- Diagnostic outputs ----
    out_dir.mkdir(parents=True, exist_ok=True)
    stem = Path(img_path).stem
    per_view = []
    for i in range(n):
        per_view.append({
            "clip": float(clip_arr[i]),
            "dino": None if dino_arr is None else float(dino_arr[i]),
            "siglip": None if siglip_arr is None else float(siglip_arr[i]),
            "iou": float(iou_arr[i]),
        })

    preview_path = out_dir / f"{stem}_preview.jpg"
    spotlight_path = out_dir / f"{stem}_spotlight.jpg"
    mesh_path = out_dir / f"{stem}.obj"

    make_preview_grid(inp_white, rgb_views, per_view, matched_idx, preview_path)
    make_spotlight(inp_white, rgb_views, matched_idx, spotlight_path)
    try:
        mesh.export(str(mesh_path))
    except Exception as exc:
        print(f"  warning: could not save mesh OBJ: {exc}")
        mesh_path = None

    return {
        "image": Path(img_path).name,
        "final_score_100": final,
        "reliability": flag,
        "review_reasons": reasons,
        "visual_score_100": visual_score_100,
        # Matched-view (primary) scores:
        "clip_score_100": matched_clip_score,
        "dino_score_100": matched_dino_score,
        "siglip_score_100": matched_siglip_score,
        "silhouette_score_100": silhouette_score_100,
        "matched_view_indices": matched_idx,
        "matched_iou_raw": matched_iou_raw,
        # Geometry + reliability diagnostics:
        "geometry_score_100": geo["geometry_score_100"],
        "consistency_penalty": consistency,
        "blob_penalty": blob_penalty,
        "fragility_penalty": fragility,
        "inter_view_diversity": diversity,
        "bootstrap_mean": boot_mean,
        "bootstrap_std": boot_std,
        # Full-view raw / per-view data:
        "clip_per_view_100": clip_arr.tolist(),
        "silhouette_per_view_100": iou_arr.tolist(),
        "dino_per_view_100": None if dino_arr is None else dino_arr.tolist(),
        "siglip_per_view_100": None if siglip_arr is None else siglip_arr.tolist(),
        "iou_per_view_raw": iou_raw_arr.tolist(),
        "view_meta": view_meta,
        # File outputs:
        "preview_path": str(preview_path),
        "spotlight_path": str(spotlight_path),
        "mesh_path": str(mesh_path) if mesh_path else None,
        **{k: v for k, v in geo.items() if k != "geometry_score_100"},
    }


def evaluate_images(image_paths, output_dir="hunyuan_eval_outputs",
                    use_dino=True, use_siglip=True, resume=True):
    logging.getLogger("hy3dgen.shapgen").setLevel(logging.ERROR)
    os.environ.setdefault("PYOPENGL_PLATFORM", "egl")
    os.environ.setdefault("PYGLET_HEADLESS", "True")

    out_dir = Path(output_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    json_path = out_dir / "hunyuan_multisignal_scores.json"

    # Resume support
    existing = {}
    if resume and json_path.exists():
        try:
            for r in json.loads(json_path.read_text(encoding="utf-8")):
                existing[r["image"]] = r
            if existing:
                print(f"resume: loaded {len(existing)} existing scores from {json_path}")
        except Exception:
            pass

    device = setup_device()
    rembg, shape_pipe, clip_model, clip_preprocess, dino, siglip_model, siglip_preprocess = \
        load_models(device, use_dino=use_dino, use_siglip=use_siglip)

    # Renderer sanity check — fail loudly here instead of silently producing
    # zero scores for every image.
    ok, coverage = render_self_test()
    print(f"renderer self-test: {'OK' if ok else 'FAILED'} (alpha coverage = {coverage:.3f})")
    if not ok:
        raise RuntimeError(
            "Offscreen renderer is producing blank frames. This is almost always "
            "a Colab EGL/OpenGL context problem. FIX: Runtime -> Restart runtime, "
            "then re-run Cell 1 (apt install must complete BEFORE pyrender is "
            "imported), then Cell 2, then Cell 3 — in that exact order."
        )

    results = list(existing.values())
    seen = set(existing.keys())

    for idx, img_path in enumerate(image_paths, 1):
        name = Path(img_path).name
        if name in seen:
            print(f"\n[{idx}/{len(image_paths)}] {name}  (cached)")
            continue
        print(f"\n[{idx}/{len(image_paths)}] {name}")
        try:
            mesh, rembg_img = make_mesh_from_image(img_path, rembg, shape_pipe, device)
            result = score_one(img_path, rembg_img, mesh,
                               clip_model, clip_preprocess,
                               dino,
                               siglip_model, siglip_preprocess,
                               device, out_dir)
        except Exception as exc:
            print(f"  ERROR: {exc}")
            continue

        results.append(result)
        seen.add(name)

        dino_str = "n/a" if result["dino_score_100"] is None else f"{result['dino_score_100']:.0f}"
        siglip_str = "n/a" if result["siglip_score_100"] is None else f"{result['siglip_score_100']:.0f}"
        print(
            f"  final={result['final_score_100']:.2f}  "
            f"reliability={result['reliability']}  "
            f"(matched_clip={result['clip_score_100']:.0f} "
            f"dino={dino_str} siglip={siglip_str} "
            f"iou={result['silhouette_score_100']:.0f}) "
            f"boot_std={result['bootstrap_std']:.1f}"
        )
        if result["review_reasons"]:
            print("  review:", ", ".join(result["review_reasons"]))
        print(f"  spotlight: {result['spotlight_path']}")

        # Stream-write so a crash mid-batch doesn't lose results.
        with open(json_path, "w", encoding="utf-8") as f:
            json.dump(results, f, ensure_ascii=False, indent=2)

        del mesh
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    # Add rank percentiles (calibration-free ranking signal).
    if results:
        scores = [r["final_score_100"] for r in results]
        order = np.argsort(np.argsort(scores))  # 0 = lowest, n-1 = highest
        n_r = len(scores)
        for i, r in enumerate(results):
            r["rank_percentile"] = float((order[i] / max(1, n_r - 1)) * 100.0)
        with open(json_path, "w", encoding="utf-8") as f:
            json.dump(results, f, ensure_ascii=False, indent=2)

    html_path = out_dir / "report.html"
    write_html_report(results, html_path)
    print("\nSaved:", json_path)
    print("Report:", html_path)
    print("\nINTERPRETATION:")
    print("  reliability=high  -> trust the score; signals agree, score is stable")
    print("  reliability=medium -> ranking probably right; absolute number may be noisy")
    print("  reliability=low   -> open the SPOTLIGHT image; signals disagree or mesh is off")
    return results

print("Multisignal evaluator loaded. Now run Cell 3.")


In [ ]:
# Cell 3: ask how many images, upload them, evaluate, then print scores one by one.
from google.colab import files
from pathlib import Path
from IPython.display import display, Image as DisplayImage

EXPECTED_IMAGES = int(input("How many images do you want to upload/evaluate? ").strip())
if EXPECTED_IMAGES <= 0:
    raise ValueError("Please enter a positive number.")

print(f"Upload exactly {EXPECTED_IMAGES} image(s). Supported: png, jpg, jpeg, webp")
uploaded = files.upload()

valid_exts = {".png", ".jpg", ".jpeg", ".webp"}
img_paths = []
for name in uploaded.keys():
    p = Path("/content/Hunyuan3D-2") / name
    if p.suffix.lower() in valid_exts:
        img_paths.append(str(p))

img_paths = sorted(img_paths)
if len(img_paths) != EXPECTED_IMAGES:
    raise RuntimeError(
        f"Expected {EXPECTED_IMAGES} image(s), but found {len(img_paths)} valid image file(s). "
        "Run this cell again and upload the right files."
    )

print("Images ready:")
for p in img_paths:
    print(" -", Path(p).name)

OUTPUT_DIR = "/content/hunyuan_eval_outputs"
results = evaluate_images(
    img_paths,
    output_dir=OUTPUT_DIR,
    use_dino=True,
    use_siglip=True,
    resume=False,
)

# Use chr(10) for literal newlines so an editor/linter cannot accidentally
# rewrite an escaped string into a multi-line literal (which would crash).
NL = chr(10)
BAR = "=" * 20
print(NL + BAR + " FINAL SCORES " + BAR)
sorted_results = sorted(results, key=lambda x: x["final_score_100"], reverse=True)
for r in sorted_results:
    dino = "n/a" if r.get("dino_score_100") is None else f"{r['dino_score_100']:.1f}"
    siglip = "n/a" if r.get("siglip_score_100") is None else f"{r['siglip_score_100']:.1f}"
    reasons = "; ".join(r.get("review_reasons", [])) or "none"
    line1 = f"{r['image']}: {r['final_score_100']:.2f}/100 | reliability={r['reliability']}"
    line2 = f"  clip={r['clip_score_100']:.1f}, dino={dino}, siglip={siglip}, iou={r['silhouette_score_100']:.1f}, geom={r['geometry_score_100']:.1f}"
    line3 = f"  review: {reasons}"
    print(line1)
    print(line2)
    print(line3)

print(NL + "Saved outputs:")
print(f" - JSON: {OUTPUT_DIR}/hunyuan_multisignal_scores.json")
print(f" - HTML report: {OUTPUT_DIR}/report.html")
print(f" - preview/spotlight images and OBJ meshes: {OUTPUT_DIR}")

print(NL + "Spotlight previews (input photo next to its top-3 matched renders):")
for r in sorted_results:
    header = f"{r['image']} | final={r['final_score_100']:.2f} | reliability={r['reliability']}"
    print(NL + header)
    if r.get("spotlight_path"):
        display(DisplayImage(filename=r["spotlight_path"]))
